# CIS 5450 Project: Difficulty Topics
**Team Members:** [*Jiajun Chen, Pai Liang, Ruiyao Liu, Xinyi Gong*]

> This notebook documents how you implemented difficulty topics in your project. Use the link button in the top right when you select a cell to get a **hyperlink**.


---

## Topic: 1: **Near-zero-variance**
**Hyperlink: 4.1 Near-zero-variance**

### Implementation & Rationale


#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Near-zero-variance filtering is a preprocessing technique that removes features whose values are almost constant across all rows. In practice we found some features are effectively constant — for example, a binary flag that is `True` for 99% of rows. Such features carry virtually no discriminative information yet still inflate dimensionality, slow down downstream models, and can destabilize coefficient estimates in linear models. The standard approach is to compute the **dominant value ratio** (proportion of the most frequent value) for each column and drop any feature whose dominant value ratio exceeds a chosen threshold.

**How we implemented it:**
After excluding identifier columns (`appid`, `name`, `developer_name`, `publisher_name`) and the leakage column `estimated_sales`, we computed `X[col].value_counts(normalize=True).iloc[0]` for every feature and compared it against `DOMINANCE_THRESHOLD = 0.98`. Any column whose most-frequent value occupied ≥ 98% of all observations was dropped, with the column name and dominant ratio printed for transparency.

**Why we used it in this project:**
The Steam dataset contains many platform/flag columns (e.g., `supports_windows`, `supports_mac`, `supports_linux`) and category indicators that are extremely lopsided — `supports_windows` is `True` for nearly every game, so it cannot help discriminate high-revenue from low-revenue titles. Because our project's stated goal is to deliver **interpretable, actionable strategies for small game studios**, every retained feature must contribute real signal that a developer can act on. Removing NZV features early keeps the correlation heatmap (Step 4.2) and the LASSO regression (Step 4.3) focused on features that actually vary across the catalog, which directly improves the readability of the final coefficient ranking and avoids wasting model capacity on near-constant flags.

---

## Topic: 2: **Correlation Filtering**
**Hyperlink: 4.2 Correlation-based Filtering**

### Implementation & Rationale


#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Correlation filtering is a multicollinearity-reduction technique. When two features are highly linearly correlated, they encode largely the same information, which (i) inflates the variance of OLS / GLM coefficient estimates, (ii) makes the sign and magnitude of those coefficients unstable and hard to interpret, and (iii) can degrade the convergence and selection behavior of regularized methods such as LASSO. The standard procedure is to compute the Pearson correlation matrix, scan its upper triangle for pairs whose `|r|` exceeds a threshold (commonly `0.85`–`0.95`), and drop one feature from each redundant pair.

**How we implemented it:**
We computed `X.corr()`, masked the upper triangle with `np.triu(..., k=1)`, and collected every pair `(f1, f2)` with `|r| > CORR_THRESHOLD = 0.85`. Rather than dropping arbitrarily, for each pair we computed `corr(f1, y)` and `corr(f2, y)` against the log-revenue target and kept the feature with the stronger correlation to `y`, dropping the weaker one.

**Why we used it in this project.**
Steam metadata may have structurally redundant fields. If we left these in, the linear regression and statistical models in Part 5 would produce unstable, hard-to-explain coefficients. By keeping the more target-correlated feature from each pair, we preserve as much predictive signal as possible while making the downstream LASSO selection cleaner — LASSO is known to behave erratically on highly collinear inputs (it picks one feature from a correlated group somewhat arbitrarily), so pre-filtering at `|r| > 0.85` makes the final selected feature set much more reproducible and interpretable.

---

## Topic: 3: **Lasso**
**Hyperlink: 4.3 LASSO Filtering**

### Implementation & Rationale


#### How did you implement this topic and why did you use it?

**Brief Introduction:**
LASSO (Least Absolute Shrinkage and Selection Operator) is a linear regression method that augments the standard squared-error loss with an L1 penalty on the coefficient vector:

$$\min_{\beta} \frac{1}{2n} \|y - X\beta\|_2^2 + \alpha \|\beta\|_1$$

The defining property of the L1 penalty is that it can drive coefficients to exactly zero, so LASSO performs continuous shrinkage and discrete feature selection in a single optimization. The regularization strength `α` controls how aggressive that selection is: larger `α` yields a sparser, more parsimonious model.

**How we implemented it:**
We first standardized all surviving features with `StandardScaler` (zero mean, unit variance) so that the magnitudes of LASSO coefficients are directly comparable across features. We then fit `LassoCV(alphas=np.logspace(-4, 1, 100), cv=5, max_iter=10000)`, which performs 5-fold cross-validation across 100 candidate `α` values and selects the one minimizing CV error. After fitting, we applied an additional pruning rule: any feature with `|coefficient| < 0.01` was treated as a negligible contributor and dropped. Finally, we plotted the surviving coefficients as a horizontal bar chart with **blue = positive effect on log-revenue** and **red = negative effect**, producing a one-glance ranking of which features actually push revenue up or down.

**Why we used it in this project.**
After the NZV and correlation steps, we still had a sizeable feature set, and we needed a principled, data-driven way to decide which features genuinely matter for predicting revenue. LASSO is ideal here because (a) it automatically performs feature selection through coefficient sparsity, (b) the cross-validated `α` removes the need to hand-tune the regularization strength, and (c) when fit on standardized inputs, the resulting coefficients double as a **direct importance ranking** — the bar chart we produced in Part 4.3 is essentially the actionable answer to *"which of these levers most strongly moves revenue?"* The resulting `X_selected` then serves as the cleaned, interpretable feature matrix that feeds directly into the OLS / statistical models in Part 5.

---

## Topic 4: **Log-scale Visualization of Heavy-tailed Revenue**
**Hyperlink: 3.2 Distribution of Revenue**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Steam revenue is one of the most extreme right-skewed variables we have ever worked with: it spans seven orders of magnitude, with a tiny number of blockbusters dominating the aggregate. A naive histogram of `revenue` collapses ~99% of the data into the first bin, so the entire shape of the distribution is invisible. The standard cure is to plot `log10(revenue)` for the strictly-positive portion of the data, which converts a power-law-like distribution into something close to bell-shaped and lets the eye actually compare tiers. We pair this with a **Lorenz-style cumulative-share curve** so that the inequality of the distribution is quantified visually, not just hinted at.

**How we implemented it:**
We filtered to `df['revenue'] > 0`, computed `np.log10(rev_pos)`, and plotted an 80-bin histogram with a red dashed line at the median. To quantify concentration we sorted revenue ascending, took `np.cumsum(rev) / rev.sum()`, and plotted that against the cumulative game-percentile, overlaying the 45° equality line for reference. Finally we printed concrete summary numbers — "Top 1% of games account for X% of revenue" — so the visual claim is anchored to a hard statistic.

**Why we used it in this project.**
The single most important fact about this dataset is that it is winner-take-all: the top 1% of games capture more than 80% of platform revenue. Every downstream modelling decision (log-transforming the target in Part 6, choosing quantile regression in addition to OLS, using HC3 robust standard errors) is a direct consequence of this finding. Without log-scaling, we would have looked at a "wall + flat line" plot and never realised that the *shape* of the distribution is the main story — and we would have chosen modelling tools (mean-squared error, OLS without robust SEs) that are quietly wrong for power-law data.

---

## Topic 5: **Feature-Lift Ratio for Binary Game Features**
**Hyperlink: 3.6 Game Features (Categories) & Revenue**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
With ~20 binary `cat_*` columns (Trading Cards, Cloud Save, Achievements, Controller Support, etc.), we needed a single comparable metric to rank which Steam features are most associated with higher revenue. Comparing raw means is misleading because revenue is skewed; comparing medians is more honest but yields absolute-dollar differences that are hard to compare across features with very different baseline rates. The fix is the **lift ratio**: `median(revenue | feature = 1) / median(revenue | feature = 0)`. A lift of 3.0 means games carrying that feature earn 3× the median of games without it. The metric is scale-free, robust to outliers, and immediately interpretable.

**How we implemented it:**
For each `cat_*` column we partitioned the dataset into "with-feature" and "without-feature" subsets, took medians of each, and computed `lift = median_with / max(median_without, 1)`. The `max(..., 1)` guards against division-by-zero when the without-feature median collapses. We filtered out features with fewer than 200 games to avoid noise, then plotted a horizontal bar chart sorted by lift, colouring bars green (lift > 1) or salmon (lift < 1) so the eye can immediately see which features hurt and which help.

**Why we used it in this project.**
The whole project is framed around what *small studios can controllably do* to improve their odds. Steam features (Trading Cards, Cloud Save, Achievements) are exactly that — they are low-cost engineering decisions, not market forces. The lift ratio gave us a clean, defensible ranking that we could carry forward into the modelling section: the same features that ranked highest on lift (Trading Cards, Cloud Save, Full Controller Support) also turned out to have the largest, most precisely-estimated coefficients in the OLS and quantile regressions in Part 6. The descriptive and inferential evidence converged, which is the strongest kind of result.

---

## Topic 6: **Multi-threshold Success-Rate Curve for Studio Segments**
**Hyperlink: 3.7 Indie / Self-Published Analysis**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Comparing "how do indie studios do versus established publishers?" with a single number (median revenue, mean revenue, percent profitable) hides the part of the story that actually matters: **the gap may be huge at one revenue level and tiny at another**. A success-rate curve fixes this by sweeping a threshold across the full revenue range and plotting, for each segment, the fraction of games that exceed the threshold. The shape of the curves — not their height at any single point — tells you whether the segment disadvantage is uniform or whether it concentrates at certain tiers.

**How we implemented it:**
We defined three segments (Small Indie, All Games, Established >20 games) and a vector of revenue thresholds (`100, 1k, 5k, 10k, 50k, 100k, 500k, 1M`). For each `(segment, threshold)` pair we computed `(subset['revenue'] >= threshold).mean()`, giving the empirical probability of clearing that bar, and plotted the resulting curves on a log-x axis so all eight thresholds get visual room. A log x-axis is essential here because the thresholds themselves span four orders of magnitude.

**Why we used it in this project.**
The single most encouraging finding for the project's audience — small studios — comes out of this plot: established publishers dominate at *every* threshold, but the **gap narrows at the highest revenue tiers**. In other words, conditional on a game being a breakout, the indie/established distinction matters less than it does at the median. This nuance would be completely invisible in a single bar chart of median revenue, and it directly motivated the framing in our final recommendations: small studios cannot beat the median outcome, but they *can* compete in the tail if they execute on the controllable levers. A medians-only comparison would have led us to write a far more pessimistic conclusion than the data actually supports.

---

## Topic 7: **PCA as a Structural Diagnostic (not a Predictive Model)**
**Hyperlink: 6.2 PCA — Understanding the structure of the selected features**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Principal Component Analysis is most often introduced as a *dimensionality reduction* technique that you feed into a downstream regressor. We deliberately use it differently: as a **diagnostic** that answers three questions before we touch a regression model. (1) Are the LASSO-selected features actually independent, or is the "effective dimensionality" much smaller than the count of columns? (2) What does each component *mean* — i.e. is there an interpretable axis like "production polish vs casual"? (3) Do high-revenue games visually separate from low-revenue games in the leading-PC plane, even though PCA itself is unsupervised? If the answer to (3) is yes, the feature set carries genuine signal about the target.

**How we implemented it:**
We `StandardScaler`-ed `X_model` (zero mean, unit variance) so that `log_price_usd` — which spans ~5 native units — would not crowd out the 0/1 indicator features. Then we fit `PCA(n_components=10)`, extracted the explained-variance ratio + cumulative curve for the scree plot, and computed the *loadings matrix* as `components_.T * sqrt(explained_variance_)` so the loadings are interpretable as feature-component correlations. The third panel projects every game onto PC1 × PC2, colours it by an ex-post revenue tier, and shows that hits cluster on one side of PC1 even though PCA never saw revenue.

**Why we used it in this project.**
Doing PCA first protects every subsequent OLS / quantile regression interpretation. If the scree plot had shown 95% of variance compressed into 1–2 components, that would have been a red flag that our LASSO-selected features were quietly redundant and our coefficient estimates would be unstable. Instead, the cumulative-variance curve climbed gradually — confirming non-redundancy, which the VIF check in 6.5 then independently corroborated. The revenue-tier scatter served as a *model-free sanity check*: before fitting any regression, we already had visual evidence that the feature set separates hits from flops, which means a regression should work.

---

## Topic 8: **OLS Linear Regression with HC3 Heteroskedasticity-Robust SEs**
**Hyperlink: 6.3 Model 1 — OLS Linear Regression**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Plain OLS produces unbiased coefficient estimates even when residual variance is non-constant, but its **standard errors** assume homoskedasticity, and when that assumption fails the t-stats, p-values, and confidence intervals are wrong — usually too narrow. For a heavy-tailed target like log-revenue, residual spread typically grows with the fitted value, so the homoskedastic SEs are exactly the wrong tool. **HC3 (MacKinnon–White)** is one of a family of heteroskedasticity-consistent estimators that re-computes the variance of `β̂` from a sandwich formula `(X'X)^-1 X' Ω X (X'X)^-1`, where `Ω` is built from the squared residuals and a leverage correction. HC3 specifically uses a stronger leverage adjustment than HC0/HC1, which makes it the most defensible choice for cross-sectional data with high-leverage observations (like the few mega-hits in our sample).

**How we implemented it:**
We assembled the design matrix with `sm.add_constant(X_model)`, log-transformed the heavy-tailed continuous predictors (`price_usd`, `pub_game_count`, `dev_game_count`) so coefficients become elasticities, centred `release_year` at 2015 so the intercept is interpretable, and called `sm.OLS(y, X_sm).fit(cov_type='HC3')`. We then built a coefficient table that includes both `coef` and `(exp(coef) - 1) * 100` so each binary-feature coefficient can be read directly as a percentage change in expected revenue.

**Why we used it in this project.**
We need *interpretable*, *defensible* effect sizes — this is the explanatory model, not a black-box predictor. Two design choices made the OLS results reliable. First, the HC3 SEs ensure that "Trading Cards has a positive, statistically significant effect" is a claim our diagnostics can actually back, even though residual spread varies across the fitted range (visible in the residuals-vs-fitted plot in 6.5). Second, the log-log transformation on `price_usd` makes the dominant coefficient (~1.3) directly interpretable as a price elasticity — a doubling of price is associated with roughly a 150% revenue increase, holding all else constant. Without the log transform, the coefficient would be a meaningless dollars-per-USD-of-price number that does not generalise across the price range.

---

## Topic 9: **Quantile Regression at Multiple τ (Hit-Amplifier Detection)**
**Hyperlink: 6.4 Model 2 — Quantile Regression**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
OLS estimates the conditional **mean** of log-revenue. For a heavy-tailed target this is informative but incomplete: it cannot distinguish between a feature that gives every game a small boost and a feature that disproportionately lifts the games at the top of the distribution. **Quantile regression** fits a separate set of coefficients at any chosen quantile τ ∈ (0,1) by minimising an asymmetrically-weighted absolute loss `ρ_τ(u) = u(τ − 1{u<0})`. Comparing the trajectory of a coefficient across τ = 0.25, 0.50, 0.75, 0.90 reveals whether a feature is a **universal driver** (flat trajectory), a **hit-amplifier** (rising trajectory), or a **floor-raiser** (falling trajectory) — three structurally different stories that mean-regression cannot tell apart.

**How we implemented it:**
We looped over `quantiles = [0.25, 0.50, 0.75, 0.90]`, fitting `QuantReg(y, X_sm).fit(q=q, max_iter=3000)` at each. The `max_iter=3000` was non-default and necessary because the linear-programming solver for quantile regression occasionally fails to converge at extreme quantiles on large samples (n ≈ 80k). We collected the coefficients into a wide DataFrame indexed by feature, columns by τ, and plotted each feature's trajectory as a connected line so the *shape* of the trajectory is visible at a glance.

**Why we used it in this project.**
This is where the analysis goes from "average effects" to "actionable advice for hit-seekers." Two findings only emerged from the τ-trajectory plot: (1) **Steam Trading Cards** is a true hit-amplifier — its coefficient at τ = 0.90 is several times its coefficient at τ = 0.25, meaning the feature is disproportionately valuable to games that could otherwise be hits, plausibly because Trading Cards correlate with marketplace presence and collector engagement that compound on already-popular titles. (2) **The "Casual" genre penalty grows worse at higher quantiles** — pure-casual positioning is not just a floor-lowerer, it is a ceiling constraint. Neither of these findings is detectable from OLS alone, and both directly shaped the priority ordering in our final recommendations table (Trading Cards = highest priority for hit-seekers; avoid pure-casual framing if aiming for the upper tiers).


---

## Topic 10: **Hypothesis Testing for Revenue Drivers**
**Hyperlink: Part 5 — Hypothesis Testing (5.1–5.4)**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Before nonlinear modeling, we tested core economic hypotheses about revenue drivers using statistical inference. This step checks whether key relationships observed in EDA hold under formal tests.

**How we implemented it:**
Using the cleaned dataset, we tested hypotheses in Part 5 (Halo effect, self-publishing effect, and price-signal effect) with appropriate group comparisons and regression-based evidence on log-revenue scale:

$$y=\log(1+\text{revenue})$$

We report effect direction, magnitude, and significance, then carry validated signals forward into model design and interpretation.

**Why we used it in this project.**
Hypothesis testing provides inferential grounding for later predictive modeling. It helps separate robust relationships from descriptive noise and ensures final recommendations are not based on visualization alone.

---

## Topic 11: **Random Forest for Nonlinear Tabular Regression**
**Hyperlink: Part 7 — Training Curves + 4-Model Comparison (Random Forest)**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
Random Forest is a bagging ensemble of decision trees. Its prediction is the average across trees:

$$\hat{f}_{RF}(x)=\frac{1}{B}\sum_{b=1}^{B}T_b(x)$$

Because each tree is trained on a bootstrap sample with random feature subsetting, tree errors are less correlated; averaging then reduces variance while preserving nonlinear interaction capacity.

**How we implemented it:**
We trained `RandomForestRegressor(n_estimators=100, random_state=42)` using the same leakage-safe feature set used across all models (post NZV + correlation + LassoCV). The target is:

$$y=\log(1+\text{revenue})$$

The split is fixed at 80/20 train-test, and evaluation is on the held-out set with:

$$RMSE=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2},\quad MAE=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

$$R^2=1-\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}$$

We kept depth and split hyperparameters at defaults on purpose so this model acts as a low-tuning baseline rather than a heavily optimized ensemble.

**Why we used it in this project.**
Random Forest establishes a robust nonlinear baseline with minimal tuning overhead. If boosted trees or neural models cannot beat it consistently, their added complexity is not justified.

---

## Topic 12: **XGBoost for Gradient-Boosted Revenue Prediction**
**Hyperlink: Part 7 — 4-Model Comparison + Modeling Conclusion (Summary of Results)**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
XGBoost is an additive boosted-tree model:

$$\hat{y}_i^{(t)}=\hat{y}_i^{(t-1)}+\eta f_t(x_i)$$

Each iteration adds a tree that improves residual fit while controlling model complexity through regularization:

$$\mathcal{L}^{(t)}=\sum_i l\big(y_i,\hat{y}_i^{(t-1)}+f_t(x_i)\big)+\Omega(f_t),\quad \Omega(f)=\gamma T+\frac{1}{2}\lambda\|w\|_2^2$$

This sequential error-correction is usually stronger than bagging on structured tabular tasks.

**How we implemented it:**
We first trained a baseline XGBoost with `n_estimators=100` and fixed seed under the same preprocessing pipeline as other models. Then we tuned only XGBoost via 3-fold CV on `R^2`, using a compact search space:

- `max_depth in {4,6,8}` to control interaction complexity.
- `learning_rate in {0.05,0.1}` to trade off step size vs stability.
- `n_estimators in {100,200}` to control boosting rounds.
- `subsample in {0.8,1.0}` to improve generalization.

The tuned configuration is selected as the final non-linear working model and passed to SHAP in Part 8.

**Why we used it in this project.**
XGBoost is both the strongest predictive candidate and the most relevant model for downstream interpretation. Concentrating tuning budget here maximizes performance gain per unit runtime and directly supports the final explanatory analysis.

---

## Topic 13: **MLP-Shallow as a Neural Baseline**
**Hyperlink: Part 7 — Training Curves + 4-Model Comparison (MLP-Shallow)**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
A shallow MLP with ReLU layers is defined by:

$$h^{(1)}=\sigma(W_1x+b_1),\quad h^{(2)}=\sigma(W_2h^{(1)}+b_2),\quad \hat{y}=W_3h^{(2)}+b_3$$

and optimized with L2-regularized MSE:

$$\min_{\theta}\;\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2+\alpha\|\theta\|_2^2$$

**How we implemented it:**
We used `MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', solver='adam', early_stopping=True, random_state=42)`. Inputs are standardized with `StandardScaler` fit on training data only (critical for gradient-based optimization), then transformed on test. Early stopping monitors validation loss and halts training when improvement stalls, reducing overfitting risk.

This setup intentionally balances expressiveness and training stability: enough capacity to model nonlinearities, but small enough to serve as a clean neural baseline.

**Why we used it in this project.**
MLP-Shallow answers whether a practical, lightly tuned neural network can match tree-based models on this feature matrix. It is the neural reference point before testing deeper architectures.

---

## Topic 14: **MLP-Deep for Capacity Stress Testing**
**Hyperlink: Part 7 — Training Curves + 4-Model Comparison (MLP-Deep)**

### Implementation & Rationale

#### How did you implement this topic and why did you use it?

**Brief Introduction:**
MLP-Deep increases representation depth while keeping the same regularized objective:

$$\min_{\theta}\;\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2+\alpha\|\theta\|_2^2$$

The core question is whether additional depth creates useful higher-order feature compositions or mainly amplifies variance on binary-heavy tabular inputs.

**How we implemented it:**
We trained `MLPRegressor(hidden_layer_sizes=(256, 128, 64, 32), alpha=1e-3, learning_rate='adaptive', batch_size=256, early_stopping=True, n_iter_no_change=20, random_state=42)` with the same split, features, and scaling pipeline as MLP-Shallow.

Parameter choices are purposeful:

- Deeper layers increase capacity for complex interactions.
- `alpha=1e-3` strengthens L2 shrinkage to counter overfitting.
- `learning_rate='adaptive'` stabilizes late-stage optimization.
- Larger `batch_size=256` smooths gradient noise.
- Longer patience (`n_iter_no_change=20`) gives the deeper network time to converge.

**Why we used it in this project.**
Comparing MLP-Deep against MLP-Shallow isolates the effect of depth under controlled preprocessing. If deeper capacity still does not outperform tree ensembles, that is a substantive finding about model suitability for this dataset.

